# TASK 3 — Data Wrangling & Statistical Analysis

In [2]:
pip install mysql-connector-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import mysql.connector
from sklearn.datasets import load_iris

In [ ]:
conn = mysql.connector.connect(
    user="root",
    password="Kajal@3105",
    host="localhost",
    database="nimbus_core_sql_db"
)
cursor = conn.cursor()

In [51]:
core_df = pd.read_csv(r"C:\Users\HP\Desktop\kajal\RoaDo_Task\customers.csv")

print(core_df.head())

   customer_id    company_name       industry  company_size  country_code  \
0            1   Rush Platform     E-commerce         small            FR   
1            2       TrekStack  Manufacturing         small            CA   
2            3      Wave Group      Education    enterprise            US   
3            4    KeenSoftware     Technology         large            GB   
4            5  QuestAnalytics      Logistics         large            US   

     country_name          timezone                contact_email  \
0          France      Europe/Paris       michael.baker@corp.net   
1          Canada   America/Toronto  emily.moore+work@startup.co   
2   United States  America/New_York  olga.robinson@wavegroup.com   
3  United Kingdom     Europe/London     lars.jackson@outlook.com   
4   United States   America/Chicago       joshua.green@yahoo.com   

    contact_name          signup_date  signup_source   is_active  \
0  Michael Baker  2023-01-31 00:00:00        partner        

In [52]:
print("Customers BEFORE cleaning:", core_df.shape[0])

Customers BEFORE cleaning: 1204


### DATA CLEANING

In [57]:
cols = [
    "customer_id", "company_name", "industry", "company_size",
    "country_code", "country_name", "timezone", "contact_email",
    "contact_name", "signup_date", "signup_source", "is_active",
    "churned_at", "churn_reason", "nps_score"
]

core_df.columns = cols[:core_df.shape[1]]

print(core_df.columns)

Index(['customer_id', 'company_name', 'industry', 'company_size',
       'country_code', 'country_name', 'timezone', 'contact_email',
       'contact_name', 'signup_date', 'signup_source', 'is_active',
       'churned_at', 'churn_reason', 'nps_score'],
      dtype='object')


In [58]:
# Handle Missing Values
core_df = core_df.replace("", np.nan)

# NPS
core_df['nps_score'] = pd.to_numeric(core_df['nps_score'], errors='coerce')
core_df['nps_score'] = core_df['nps_score'].fillna(core_df['nps_score'].median())

# Churn reason
core_df['churn_reason'] = core_df['churn_reason'].fillna("Unknown")

In [59]:
# Fix Data Types
core_df['customer_id'] = pd.to_numeric(core_df['customer_id'], errors='coerce')
core_df['is_active'] = pd.to_numeric(core_df['is_active'], errors='coerce')

core_df['signup_date'] = pd.to_datetime(core_df['signup_date'], errors='coerce')
core_df['churned_at'] = pd.to_datetime(core_df['churned_at'], errors='coerce')

In [60]:
core_df['company_name'] = core_df['company_name'].astype(str).str.strip()
core_df['country_name'] = core_df['country_name'].astype(str).str.strip()
core_df['timezone'] = core_df['timezone'].astype(str).str.strip()

In [61]:
# Remove Duplicates
core_df = core_df.drop_duplicates()

In [62]:
# Handle Outliers (NPS)
core_df = core_df[
    (core_df['nps_score'] >= 0) &
    (core_df['nps_score'] <= 10)
]

In [63]:
# FEATURE ENGINEERING
core_df['lifetime_days'] = (
    core_df['churned_at'].fillna(pd.Timestamp.today()) - core_df['signup_date']
).dt.days

### HYPOTHESIS TEST

In [64]:
# Higher NPS → Lower churn
churned = core_df[core_df['is_active'] == 0]['nps_score']
active = core_df[core_df['is_active'] == 1]['nps_score']

print("Mean churned:", churned.mean())
print("Mean active:", active.mean())

Mean churned: 5.018796992481203
Mean active: 6.844017094017094


### SEGMENTATION

In [66]:
def segment(nps):
    if nps >= 9:
        return "Promoters"
    elif nps >= 7:
        return "Passives"
    else:
        return "Detractors"

core_df['segment'] = core_df['nps_score'].apply(segment)

In [67]:
# Segment Summary
summary = core_df.groupby('segment').agg({
    'customer_id': 'count',
    'nps_score': 'mean',
    'is_active': 'mean'
})

print(summary)

            customer_id  nps_score  is_active
segment                                      
Detractors          332   3.379518   0.605422
Passives            700   7.155714   0.830946
Promoters           172   9.441860   0.901163
